<a href="https://colab.research.google.com/github/FreddyForero1988/FreddyForero1988.github.io/blob/main/ADL_R2_A3_S11_AVANCE_DE_PROYECTO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master"
ratings = pd.read_csv(f"{BASE}/ratings.csv")
books = pd.read_csv(f"{BASE}/books.csv")

print(ratings.shape, books.shape)
print(ratings.head())
print(books.columns.tolist())

(5976479, 3) (10000, 23)
   user_id  book_id  rating
0        1      258       5
1        2     4081       4
2        2      260       5
3        2     9296       5
4        2     2318       3
['book_id', 'goodreads_book_id', 'best_book_id', 'work_id', 'books_count', 'isbn', 'isbn13', 'authors', 'original_publication_year', 'original_title', 'title', 'language_code', 'average_rating', 'ratings_count', 'work_ratings_count', 'work_text_reviews_count', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url', 'small_image_url']


In [ ]:
tags = pd.read_csv(f"{BASE}/tags.csv")
book_tags = pd.read_csv(f"{BASE}/book_tags.csv")
print(tags.shape, tags.columns.tolist())
print(book_tags.shape, book_tags.columns.tolist())

# etiquetas más usadas
top = (book_tags.merge(tags, on="tag_id")
       .groupby("tag_name")["count"].sum()
       .sort_values(ascending=False).head(30))
print(top)

# verificar que los IDs enlazan
print(book_tags["goodreads_book_id"].isin(books["goodreads_book_id"]).mean())
print(ratings["book_id"].isin(books["book_id"]).mean())

(34252, 2) ['tag_id', 'tag_name']
(999912, 3) ['goodreads_book_id', 'tag_id', 'count']
tag_name
to-read               140718761
currently-reading       7507958
favorites               4503173
fiction                 3688819
fantasy                 3548157
young-adult             1848306
classics                1756920
books-i-own             1317235
romance                 1231926
owned                   1224279
ya                       898334
mystery                  872282
non-fiction              857901
historical-fiction       815421
series                   782637
science-fiction          703866
sci-fi                   597325
paranormal               542559
kindle                   506882
contemporary             486001
book-club                446685
library                  429171
favourites               423289
owned-books              417384
horror                   415467
default                  405326
audiobook                388991
urban-fantasy            374689
nonficti

In [ ]:
STOP = {"to-read","currently-reading","favorites","favourites","books-i-own",
        "owned","owned-books","kindle","library","to-buy","default",
        "book-club","audiobook","series"}
SYN = {"ya":"young-adult","sci-fi":"science-fiction","nonfiction":"non-fiction"}

bt = book_tags.merge(tags, on="tag_id")
bt["tag_name"] = bt["tag_name"].replace(SYN)
bt = bt[~bt["tag_name"].isin(STOP)]

top100 = (bt.groupby("tag_name")["count"].sum()
          .sort_values(ascending=False).head(100))
print(top100.to_string())

tag_name
fiction                 3688819
fantasy                 3548157
young-adult             2746640
classics                1756920
science-fiction         1301191
romance                 1231926
non-fiction             1228950
mystery                  872282
historical-fiction       815421
paranormal               542559
contemporary             486001
horror                   415467
urban-fantasy            374689
adult                    337222
classic                  334113
childrens                315716
graphic-novels           313560
thriller                 309276
audiobooks               307154
ebook                    302571
vampires                 300730
adventure                295933
history                  291366
dystopian                281487
audio                    264746
historical               263267
humor                    262340
novels                   260327
chick-lit                251072
literature               249441
dystopia                 231846

In [ ]:
GENRE_MAP = {
 "fiction": ["fiction","novel","novels","general-fiction"],
 "non-fiction": ["non-fiction","nonfiction"],
 "fantasy": ["fantasy"],
 "urban-fantasy": ["urban-fantasy"],
 "science-fiction": ["science-fiction","sci-fi","scifi"],
 "sci-fi-fantasy": ["sci-fi-fantasy","fantasy-sci-fi"],
 "young-adult": ["young-adult","ya","ya-fiction","teen"],
 "new-adult": ["new-adult"],
 "middle-grade": ["middle-grade"],
 "children": ["children","childrens","children-s","children-s-books",
              "childrens-books","kids","picture-books"],
 "classics": ["classics","classic"],
 "romance": ["romance","contemporary-romance","historical-romance"],
 "paranormal-romance": ["paranormal-romance"],
 "paranormal": ["paranormal","supernatural","vampires","vampire"],
 "mystery": ["mystery","mystery-thriller"],
 "thriller": ["thriller","suspense"],
 "crime": ["crime"],
 "historical-fiction": ["historical-fiction","historical"],
 "horror": ["horror"],
 "dystopian": ["dystopian","dystopia"],
 "contemporary": ["contemporary","contemporary-fiction","realistic-fiction"],
 "literary-fiction": ["literary-fiction","literature"],
 "adventure": ["adventure"],
 "humor": ["humor"],
 "chick-lit": ["chick-lit"],
 "graphic-novels": ["graphic-novels","graphic-novel","comics"],
 "manga": ["manga"],
 "memoir": ["memoir"],
 "biography": ["biography"],
 "history": ["history"],
 "science": ["science"],
 "philosophy": ["philosophy"],
 "psychology": ["psychology"],
 "poetry": ["poetry"],
 "drama": ["drama","plays"],
 "short-stories": ["short-stories"],
 "mythology": ["mythology"],
}
tag2genre = {t: g for g, ts in GENRE_MAP.items() for t in ts}

bt = book_tags.merge(tags, on="tag_id")
bt["genre"] = bt["tag_name"].map(tag2genre)
bt = bt.dropna(subset=["genre"])

bg = bt.groupby(["goodreads_book_id","genre"], as_index=False)["count"].sum()
bg = (bg.sort_values(["goodreads_book_id","count"], ascending=[True,False])
        .groupby("goodreads_book_id").head(5))

print("cobertura de libros:", books["goodreads_book_id"].isin(bg["goodreads_book_id"]).mean())
print(bg.groupby("goodreads_book_id").size().describe())
print(bg["genre"].value_counts().to_string())

cobertura de libros: 1.0
count    10000.00000
mean         4.88080
std          0.49761
min          1.00000
25%          5.00000
50%          5.00000
75%          5.00000
max          5.00000
dtype: float64
genre
fiction               7686
fantasy               3225
contemporary          3024
romance               2920
young-adult           2569
mystery               2043
historical-fiction    2012
classics              1991
non-fiction           1803
thriller              1649
literary-fiction      1625
science-fiction       1441
adventure             1256
children              1185
chick-lit             1173
crime                 1145
paranormal            1139
history               1117
humor                 1107
sci-fi-fantasy         885
biography              833
drama                  749
philosophy             673
memoir                 661
horror                 634
psychology             494
science                494
urban-fantasy          490
graphic-novels         468
dys